# FHIR Gold analytics tables

Create analytics-friendly Gold Delta tables from current Silver records. Common FHIR fields are flattened for reporting, while resource IDs, source update timestamps, content hashes, and SCD2 validity metadata are retained. The Warehouse views are created separately by `warehouse/create_gold_views.sql` after copying these tables into the Fabric Warehouse.

In [ ]:
from pyspark.sql import functions as F

RESOURCE_TYPES = ["Patient", "Encounter", "Observation", "Condition"]
spark.conf.set("spark.sql.session.timeZone", "UTC")

def table_exists(name):
    return spark.catalog.tableExists(name)

def j(path):
    return F.get_json_object(F.col("resource_json"), path)

def write_gold(df, table_name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table_name))

In [ ]:
for resource_type in RESOURCE_TYPES:
    silver_table = f"silver.{resource_type}"
    gold_table = f"gold.{resource_type}"
    if not table_exists(silver_table):
        raise RuntimeError(f"Missing {silver_table}. Run the Silver transformation notebook first.")
    current = spark.table(silver_table).filter(F.col("is_current") == True)

    if resource_type == "Patient":
        result = current.select(
            F.col("resource_id").alias("patient_id"),
            j("$.active").cast("boolean").alias("active"),
            j("$.gender").alias("gender"),
            F.to_date(j("$.birthDate")).alias("birth_date"),
            j("$.name[0].family").alias("family_name"),
            j("$.name[0].given[0]").alias("given_name"),
            j("$.managingOrganization.reference").alias("managing_organization_reference"),
            j("$.deceasedBoolean").cast("boolean").alias("deceased_boolean"),
            "source_last_updated", "content_hash", "valid_from", "valid_to", "is_current", "ingestion_run_id", "saved_timestamp")
    elif resource_type == "Encounter":
        subject_ref = j("$.subject.reference")
        result = current.select(
            F.col("resource_id").alias("encounter_id"), j("$.status").alias("status"),
            j("$.class.code").alias("class_code"), j("$.class.display").alias("class_display"),
            j("$.type[0].coding[0].code").alias("encounter_type_code"),
            j("$.type[0].coding[0].display").alias("encounter_type_display"),
            subject_ref.alias("subject_reference"),
            F.regexp_extract(subject_ref, r"Patient/(.+)$", 1).alias("patient_id"),
            j("$.period.start").alias("period_start"), j("$.period.end").alias("period_end"),
            j("$.serviceProvider.reference").alias("service_provider_reference"),
            "source_last_updated", "content_hash", "valid_from", "valid_to", "is_current", "ingestion_run_id", "saved_timestamp")
    elif resource_type == "Observation":
        subject_ref = j("$.subject.reference")
        effective = F.coalesce(j("$.effectiveDateTime"), j("$.effectivePeriod.start"), j("$.issued"))
        result = current.select(
            F.col("resource_id").alias("observation_id"), j("$.status").alias("status"),
            j("$.code.coding[0].code").alias("code"), j("$.code.coding[0].display").alias("code_display"),
            j("$.category[0].coding[0].display").alias("category"), subject_ref.alias("subject_reference"),
            F.regexp_extract(subject_ref, r"Patient/(.+)$", 1).alias("patient_id"),
            j("$.encounter.reference").alias("encounter_reference"), effective.alias("effective_datetime"),
            j("$.valueQuantity.value").cast("double").alias("value_quantity"),
            j("$.valueQuantity.unit").alias("value_unit"), j("$.valueString").alias("value_string"),
            j("$.interpretation[0].coding[0].display").alias("interpretation"),
            "source_last_updated", "content_hash", "valid_from", "valid_to", "is_current", "ingestion_run_id", "saved_timestamp")
    else:  # Condition
        subject_ref = j("$.subject.reference")
        result = current.select(
            F.col("resource_id").alias("condition_id"),
            j("$.clinicalStatus.coding[0].code").alias("clinical_status"),
            j("$.verificationStatus.coding[0].code").alias("verification_status"),
            j("$.category[0].coding[0].display").alias("category"),
            j("$.code.coding[0].code").alias("code"), j("$.code.coding[0].display").alias("code_display"),
            subject_ref.alias("subject_reference"),
            F.regexp_extract(subject_ref, r"Patient/(.+)$", 1).alias("patient_id"),
            j("$.encounter.reference").alias("encounter_reference"),
            F.coalesce(j("$.onsetDateTime"), j("$.recordedDate")).alias("onset_datetime"),
            j("$.recordedDate").alias("recorded_date"),
            "source_last_updated", "content_hash", "valid_from", "valid_to", "is_current", "ingestion_run_id", "saved_timestamp")

    write_gold(result, gold_table)
    print(resource_type, "->", gold_table, "rows:", spark.table(gold_table).count())

## Gold output tables

- `gold.patient`
- `gold.encounter`
- `gold.observation`
- `gold.condition`

These are current-state analytics tables. Historical versions remain in Silver. The Fabric Warehouse views are defined in `warehouse/create_gold_views.sql`.